# Deepfake Detection — Colab Training
Trains all three models on a T4/A100 GPU. Runtime: ~30 min on T4.

**Before running:** Runtime → Change runtime type → T4 GPU

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND — change runtime to GPU')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 1. Clone repo & install dependencies

In [ ]:
!git clone https://github.com/mfahad16405/DeepfakeDetection.git
%cd DeepfakeDetection
!pip install timm opencv-python plotly scikit-learn grad-cam tqdm requests -q

## 2. Download Kaggle dataset
Get your API key: kaggle.com → Settings → API → **Create New Token** → upload `kaggle.json` when prompted.

In [ ]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -q
!unzip -q 140k-real-and-fake-faces.zip
!ls real_vs_fake/real-vs-fake/

DATASET_ROOT = 'real_vs_fake/real-vs-fake'

## 3. Populate image folders (5k train / 1k val / 1k test per class)

In [ ]:
import shutil
from pathlib import Path

SRC    = Path(DATASET_ROOT)
DST    = Path('images')
COUNTS = {'train': 5000, 'valid': 1000, 'test': 1000}
MAP    = {'valid': 'val'}

for src_split, n in COUNTS.items():
    dst_split = MAP.get(src_split, src_split)
    for cls in ('real', 'fake'):
        src_dir = SRC / src_split / cls
        dst_dir = DST / dst_split / cls
        dst_dir.mkdir(parents=True, exist_ok=True)
        files_list = sorted(src_dir.iterdir())[:n]
        for f in files_list:
            shutil.copy(f, dst_dir / f.name)
        print(f'  {dst_split}/{cls}: {len(files_list)} images')

## 4. Set batch size for Colab T4 (16 GB VRAM)

In [ ]:
config_text = open('src/config.py').read()
config_text = config_text.replace('BATCH_SIZE_TRAIN    = 16', 'BATCH_SIZE_TRAIN    = 32')
open('src/config.py', 'w').write(config_text)
print('Batch size set to 32')

## 5. Train all three models

In [ ]:
!python scripts/train.py --model xception

In [ ]:
!python scripts/train.py --model vit_small_patch16_224

In [ ]:
!python scripts/train.py --model efficientnet_b4

## 6. Download checkpoints
Place the downloaded `.pth` files in `checkpoints/` and push to GitHub with Git LFS.

In [ ]:
from google.colab import files
import os

for name in ['xception_best.pth', 'vit_small_patch16_224_best.pth', 'efficientnet_b4_best.pth']:
    path = f'checkpoints/{name}'
    if os.path.exists(path):
        print(f'Downloading {name}...')
        files.download(path)
    else:
        print(f'MISSING: {path} — training may have failed')

print('\nAlso downloading training log...')
files.download('output/training_log.txt')